In [ ]:
# Optional install
# !pip install shap joblib

%matplotlib inline


In [ ]:
import os
import warnings
from typing import List, Dict, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 600,
})

def resolve_path(*candidates: str) -> str:
    roots = [".", "/mnt/data"]
    for p in candidates:
        if not p:
            continue
        if os.path.exists(p):
            return p
        for root in roots:
            full = os.path.join(root, p)
            if os.path.exists(full):
                return full
    raise FileNotFoundError(f"Cannot find any of: {candidates}")

def resolve_optional_path(*candidates: str):
    try:
        return resolve_path(*candidates)
    except FileNotFoundError:
        return None

def ensure_dir(dirname: str) -> str:
    for root in [".", "/mnt/data"]:
        p = os.path.join(root, dirname)
        try:
            os.makedirs(p, exist_ok=True)
            return p
        except Exception:
            pass
    os.makedirs(dirname, exist_ok=True)
    return dirname

def numeric_clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

SAVE_DIR = ensure_dir("figures_jacs")
print("Output directory:", SAVE_DIR)


In [ ]:
# Load artifacts and identify the retained physical descriptors

PRE3_PATH = resolve_path("feat_out/preprocessing_3d.pkl")
ENS3_PATH = resolve_path("final_models/final_best_3d_ensemble.pkl")
TRAIN_XFINAL_PATH = resolve_path("feat_out/train_with_Xfinal.csv")

RAW2D_PATH = resolve_optional_path("feat_out/train_phys_2d.csv")
RAW3D_PATH = resolve_optional_path("feat_out/train_phys_3d.csv")

pre3 = joblib.load(PRE3_PATH)
ens3 = joblib.load(ENS3_PATH)

if not isinstance(pre3, dict):
    raise TypeError("preprocessing_3d.pkl is not a dict-like artifact.")
if not isinstance(ens3, dict):
    raise TypeError("final_best_3d_ensemble.pkl is not a dict-like artifact.")

selected_phys_features = list(pre3.get("selected_phys_features", []))
ensemble_feat_cols = list(ens3.get("feat_cols", []))
models = list(ens3.get("models", []))
model_name = str(ens3.get("model", "unknown"))

if len(selected_phys_features) == 0:
    raise ValueError("No selected_phys_features found in preprocessing_3d.pkl")
if len(ensemble_feat_cols) == 0:
    raise ValueError("No feat_cols found in final_best_3d_ensemble.pkl")
if len(models) == 0:
    raise ValueError("No models found in final_best_3d_ensemble.pkl")

print(f"Retained physical descriptors from preprocessing_3d.pkl: {len(selected_phys_features)}")
print(selected_phys_features)


In [ ]:
# Build the raw physical-descriptor matrix used for the correlation heatmap

raw_blocks = []
for label, path in [("2D", RAW2D_PATH), ("3D", RAW3D_PATH)]:
    if path is not None:
        df = pd.read_csv(path)
        df = numeric_clean(df)
        raw_blocks.append(df)
        print(f"Loaded {label} raw matrix: {path} | shape={df.shape}")

if len(raw_blocks) > 0:
    raw_phys_df = pd.concat(raw_blocks, axis=1)
    raw_phys_df = raw_phys_df.loc[:, ~raw_phys_df.columns.duplicated()]
    source_label = "train_phys_2d + train_phys_3d"
else:
    raw_phys_df = pd.read_csv(TRAIN_XFINAL_PATH)
    raw_phys_df = numeric_clean(raw_phys_df)
    source_label = "train_with_Xfinal (fallback)"
    print("Raw 2D/3D physical matrices not found. Falling back to train_with_Xfinal.csv")

missing_in_raw = [c for c in selected_phys_features if c not in raw_phys_df.columns]
if missing_in_raw:
    print("These retained physical descriptors are absent from the raw matrix and will be skipped:")
    print(missing_in_raw)

raw_phys_df = raw_phys_df[[c for c in selected_phys_features if c in raw_phys_df.columns]].copy()
raw_phys_df = numeric_clean(raw_phys_df)

all_nan_cols = [c for c in raw_phys_df.columns if raw_phys_df[c].notna().sum() == 0]
constant_cols = [c for c in raw_phys_df.columns if raw_phys_df[c].nunique(dropna=True) <= 1]

effective_phys_cols = [
    c for c in raw_phys_df.columns
    if c not in set(all_nan_cols + constant_cols)
]

plot_phys_df = raw_phys_df[effective_phys_cols].copy()

print(f"Raw matrix source: {source_label}")
print(f"Retained physical descriptors requested: {len(selected_phys_features)}")
print(f"Effective descriptors after dropping all-NaN / constant columns: {len(effective_phys_cols)}")

if all_nan_cols:
    print("\nDropped all-NaN descriptors:")
    print(all_nan_cols)
if constant_cols:
    print("\nDropped constant or near-constant descriptors:")
    print(constant_cols)

plot_phys_df.head()


In [ ]:
# Reorder descriptors by category to improve readability

CATEGORY_ORDER = {
    "surface_hydrophobicity": [
        "LabuteASA", "MolMR", "LogP", "ConfEnergy", "MolVol"
    ],
    "shape_3d": [
        "PMI1", "PMI2", "PMI3", "NPR1", "NPR2", "RadiusGyration",
        "Asphericity", "Eccentricity", "SpherocityIndex", "PBF",
        "InertialShapeFactor", "TPSA_3Dproxy"
    ],
    "polarity_hbond_charge": [
        "TPSA", "HBA", "HBD", "NHOHCount", "NOCount",
        "MinPartialCharge", "MaxPartialCharge",
        "MinAbsPartialCharge", "MaxAbsPartialCharge"
    ],
    "ring_count_topology": [
        "FractionCSP3", "Ring", "NumAliphaticRings", "NumAromaticRings",
        "NumAmideBonds", "AromaticRing", "has_3d",
        "NumAliphaticCarbocycles", "NumAromaticCarbocycles"
    ],
}

def reorder_features(features: List[str]) -> List[str]:
    ordered = []
    seen = set()
    for _, group in CATEGORY_ORDER.items():
        for f in group:
            if f in features and f not in seen:
                ordered.append(f)
                seen.add(f)
    for f in features:
        if f not in seen:
            ordered.append(f)
            seen.add(f)
    return ordered

ordered_phys_cols = reorder_features(effective_phys_cols)
plot_phys_df = plot_phys_df[ordered_phys_cols]

print("Ordered effective physical descriptors:")
print(ordered_phys_cols)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

# corr 为已经算好的相关矩阵 DataFrame
# corr = ...

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# 只对强相关项标数值
annot_df = corr.copy()
annot_df = annot_df.applymap(lambda x: f"{x:.2f}" if pd.notna(x) and abs(x) >= 0.60 else "")

# 更适合论文展示的柔和 diverging 配色
cmap = sns.diverging_palette(
    240,   # blue side
    15,    # red side
    s=85,  # saturation
    l=45,  # lightness
    center="light",
    as_cmap=True
)

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

plt.figure(figsize=(10.5, 9))
ax = sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    norm=norm,
    square=True,
    linewidths=0.6,
    linecolor="#F2F2F2",
    annot=annot_df,
    fmt="",
    annot_kws={"size": 8, "color": "black"},
    cbar_kws={
        "label": "Spearman correlation",
        "shrink": 0.88,
        "pad": 0.03
    }
)

ax.set_title("Correlation matrix of retained physical descriptors", fontsize=16, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Compute ensemble SHAP values on the full final input matrix,
# then restrict display to retained physical descriptors

import shap

train_df = pd.read_csv(TRAIN_XFINAL_PATH)
missing_in_train = [c for c in ensemble_feat_cols if c not in train_df.columns]
if missing_in_train:
    raise KeyError(f"train_with_Xfinal.csv is missing features, e.g. {missing_in_train[:10]}")

X_df = numeric_clean(train_df[ensemble_feat_cols].copy()).fillna(0.0)
X = X_df.values.astype(np.float32)

phys_idx = [i for i, c in enumerate(ensemble_feat_cols) if c in effective_phys_cols]
phys_names = [ensemble_feat_cols[i] for i in phys_idx]

if len(phys_idx) == 0:
    raise ValueError("None of the effective retained physical descriptors are present in feat_cols.")

# Optional subsampling if the matrix is large
MAX_SAMPLES_FOR_SHAP = 500
if X_df.shape[0] > MAX_SAMPLES_FOR_SHAP:
    X_df_shap = X_df.sample(MAX_SAMPLES_FOR_SHAP, random_state=42)
else:
    X_df_shap = X_df.copy()

X_shap = X_df_shap.values.astype(np.float32)

shap_sum = None
base_sum = None

for k, model in enumerate(models, start=1):
    if model_name.upper() == "XGB" or hasattr(model, "get_booster"):
        explainer = shap.TreeExplainer(model)
        try:
            shap_vals = explainer.shap_values(X_shap, check_additivity=False)
        except TypeError:
            shap_vals = explainer.shap_values(X_shap)
        base_val = explainer.expected_value
    else:
        explainer = shap.Explainer(model, X_df_shap)
        exp = explainer(X_df_shap)
        shap_vals = exp.values
        base_val = exp.base_values

    shap_vals = np.asarray(shap_vals, dtype=np.float64)
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[..., 0]

    if np.ndim(base_val) == 0:
        base_arr = np.full(X_shap.shape[0], float(base_val), dtype=np.float64)
    else:
        base_arr = np.asarray(base_val, dtype=np.float64).reshape(-1)
        if base_arr.size == 1:
            base_arr = np.full(X_shap.shape[0], float(base_arr[0]), dtype=np.float64)

    shap_sum = shap_vals if shap_sum is None else shap_sum + shap_vals
    base_sum = base_arr if base_sum is None else base_sum + base_arr
    print(f"Computed SHAP for ensemble member {k}/{len(models)}")

shap_mean = shap_sum / len(models)
base_mean = base_sum / len(models)

X_phys = X_df_shap.iloc[:, phys_idx].copy()
shap_phys = shap_mean[:, phys_idx]

# rank by mean absolute SHAP and keep top 15 for the main figure
mean_abs_shap = np.abs(shap_phys).mean(axis=0)
rank_df = pd.DataFrame({
    "feature": phys_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

top_n = min(15, len(rank_df))
top_features = rank_df.head(top_n)["feature"].tolist()
top_idx = [phys_names.index(f) for f in top_features]

X_phys_top = X_phys.iloc[:, top_idx].copy()
shap_phys_top = shap_phys[:, top_idx]

rank_df.head(20)


In [ ]:
# ========= 读取去冗余后的物理特征 =========
pre3 = joblib.load(PREPROCESS_PATH)

if not isinstance(pre3, dict):
    raise TypeError("preprocessing_3d.pkl 不是 dict，请检查保存格式。")

if "selected_phys_features" not in pre3:
    raise KeyError("preprocessing_3d.pkl 中未找到 'selected_phys_features'。")

retained_phys_features = list(pre3["selected_phys_features"])
print("去冗余后保留的物理特征数：", len(retained_phys_features))
print(retained_phys_features)

In [ ]:
# ========= 读取原始 2D + 3D 物理特征矩阵 =========
phys2d = pd.read_csv(PHYS2D_PATH)
phys3d = pd.read_csv(PHYS3D_PATH)

# 横向拼接
raw_phys_df = pd.concat([phys2d, phys3d], axis=1)

# 去掉重复列（如果 2D/3D 文件里有重名列）
raw_phys_df = raw_phys_df.loc[:, ~raw_phys_df.columns.duplicated()].copy()

print("raw_phys_df shape:", raw_phys_df.shape)
print("前几个列名：", raw_phys_df.columns[:10].tolist())

In [ ]:
# ========= 去掉 has_3d 及类似 indicator-like 特征 =========
EXCLUDE_HEATMAP_COLS = {"has_3d"}

def is_indicator_like(series: pd.Series) -> bool:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return False
    uniq = sorted(pd.unique(s))
    return len(uniq) <= 2 and set(uniq).issubset({0, 1, 0.0, 1.0})

# 先取去冗余后的物理特征中，在原始矩阵里实际存在的列
heatmap_cols = [c for c in retained_phys_features if c in raw_phys_df.columns]

missing_cols = [c for c in retained_phys_features if c not in raw_phys_df.columns]
if missing_cols:
    print("以下 retained 特征未在原始物理矩阵中找到：")
    print(missing_cols)

# 去掉明确排除的列
heatmap_cols = [c for c in heatmap_cols if c not in EXCLUDE_HEATMAP_COLS]

heat_df = raw_phys_df[heatmap_cols].copy()

# 强制转数值
for c in heat_df.columns:
    heat_df[c] = pd.to_numeric(heat_df[c], errors="coerce")

heat_df = heat_df.replace([np.inf, -np.inf], np.nan)

# 去掉全 NaN 列
all_nan_cols = heat_df.columns[heat_df.isna().all()].tolist()
if all_nan_cols:
    print("去掉全 NaN 列：", all_nan_cols)
heat_df = heat_df.dropna(axis=1, how="all")

# 去掉常数列
constant_cols = [c for c in heat_df.columns if heat_df[c].nunique(dropna=True) <= 1]
if constant_cols:
    print("去掉常数列：", constant_cols)
heat_df = heat_df.drop(columns=constant_cols)

# 去掉 indicator-like 列
indicator_cols = [c for c in heat_df.columns if is_indicator_like(heat_df[c])]
if indicator_cols:
    print("去掉 indicator-like 列：", indicator_cols)
heat_df = heat_df.drop(columns=indicator_cols)

print("最终用于热力图的特征数：", heat_df.shape[1])
print(heat_df.columns.tolist())

In [ ]:
# ========= 按类别重排特征顺序（更适合论文展示） =========
surface_hydrophobic = [
    "LabuteASA", "MolMR", "LogP", "ConfEnergy"
]

shape_3d = [
    "PMI1", "PMI2", "NPR1", "NPR2", "RadiusGyration",
    "SpherocityIndex", "PBF", "InertialShapeFactor"
]

polarity_charge = [
    "TPSA", "HBA", "NHOHCount", "MinPartialCharge", "MaxPartialCharge"
]

topology_counts = [
    "FractionCSP3", "Ring", "NumAliphaticRings", "NumAmideBonds", "AromaticRing"
]

preferred_order = surface_hydrophobic + shape_3d + polarity_charge + topology_counts

ordered_cols = [c for c in preferred_order if c in heat_df.columns]
remaining_cols = [c for c in heat_df.columns if c not in ordered_cols]
ordered_cols = ordered_cols + remaining_cols

heat_df = heat_df[ordered_cols].copy()

print("最终绘图顺序：")
print(ordered_cols)

In [ ]:
# ========= 计算 Spearman 相关矩阵 =========
corr = heat_df.corr(method="spearman")

print("相关矩阵维度：", corr.shape)
corr.head()

In [ ]:
# ========= 绘制优化后的热力图 =========
sns.set_theme(style="white", font_scale=1.1)

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# 只标注强相关
annot_df = corr.copy()
annot_df = annot_df.applymap(lambda x: f"{x:.2f}" if pd.notna(x) and abs(x) >= 0.60 else "")

# 更柔和、论文更友好的蓝-白-红配色
cmap = sns.diverging_palette(
    240,   # 蓝
    15,    # 红
    s=85,
    l=45,
    center="light",
    as_cmap=True
)

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

fig, ax = plt.subplots(figsize=(11, 9.5))

hm = sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    norm=norm,
    square=True,
    linewidths=0.6,
    linecolor="#F2F2F2",
    annot=annot_df,
    fmt="",
    annot_kws={"size": 12, "color": "black"},
    cbar_kws={
        "label": "Spearman correlation",
        "shrink": 0.88,
        "pad": 0.03
    },
    ax=ax
)

ax.set_title("Correlation matrix of physical descriptors", fontsize=16, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=14)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=14)

plt.tight_layout()

plt.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()

print("已保存：")
print(OUT_PNG)
print(OUT_PDF)

In [ ]:
# ========= 查看最终参与热力图的特征 =========
final_heatmap_features = corr.columns.tolist()
final_heatmap_features

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("figures_jacs", exist_ok=True)
os.makedirs("tables", exist_ok=True)

# ---- inputs ----
# 2D 全库排序取Top6
df_full2d = pd.read_csv("screening_out/stage1_2d_full_ranking_with_smiles.csv")
top6_2d = df_full2d.head(6)[["lib_idx","smiles","score","pred_mean_log","pred_std_log"]].copy()
top6_2d = top6_2d.rename(columns={"score":"score_2d"})

# 3D Top1000 scored（文件名不变）
scored3_path = "screening_out/top1000_stage2_3d_scored_with_smiles_filtered.csv"
if not os.path.exists(scored3_path):
    scored3_path = "screening_out/top1000_stage2_3d_scored_with_smiles.csv"
df3 = pd.read_csv(scored3_path)

# 数值化并剔除失败点
for c in ["score","score_3d","pred_mean_log_3d","pred_std_log_3d"]:
    if c in df3.columns:
        df3[c] = pd.to_numeric(df3[c], errors="coerce")
df3_ok = df3[df3["score_3d"].notna()].copy()

# 重点：读取文件名不变，读取后截取前6行实现3D Top6
top6_3d = pd.read_csv("screening_out/top10_final_from_3d_with_smiles.csv").head(6)[
    ["lib_idx","smiles","score_3d","pred_mean_log_3d","pred_std_log_3d"]
].copy()

# ---- evaluate 2D-Top6 by 3D scores ----
top6_2d_eval = top6_2d.merge(
    df3_ok[["lib_idx","smiles","score_3d","pred_mean_log_3d","pred_std_log_3d"]],
    on=["lib_idx","smiles"],
    how="left"
)

missing = int(top6_2d_eval["score_3d"].isna().sum())
print("2D-Top6 missing 3D scores (not in Top1000 or failed):", missing)

# 提取可用分数
vals_2d = top6_2d_eval["score_3d"].dropna().values
vals_3d = top6_3d["score_3d"].dropna().values

# ---- 统计表格 ----
stats = pd.DataFrame([
    {
        "group":"2D-Top6 (evaluated by 3D score)",
        "n": int(len(vals_2d)),
        "mean_3d_score": float(np.mean(vals_2d)) if len(vals_2d)>0 else np.nan,
        "median_3d_score": float(np.median(vals_2d)) if len(vals_2d)>0 else np.nan,
        "min_3d_score": float(np.min(vals_2d)) if len(vals_2d)>0 else np.nan,
        "max_3d_score": float(np.max(vals_2d)) if len(vals_2d)>0 else np.nan,
    },
    {
        "group":"3D-Top6 (final shortlist)",
        "n": int(len(vals_3d)),
        "mean_3d_score": float(np.mean(vals_3d)),
        "median_3d_score": float(np.median(vals_3d)),
        "min_3d_score": float(np.min(vals_3d)),
        "max_3d_score": float(np.max(vals_3d)),
    }
])
stats.to_csv("tables/top6_2d_vs_3d_score3d_stats.csv", index=False, encoding="utf-8")
print("Saved stats:", "tables/top6_2d_vs_3d_score3d_stats.csv")
print(stats)

# ---- 绘图：箱线+散点 ----
plt.figure(figsize=(5.2, 4.2))

data = [vals_2d, vals_3d]
labels = [f"2D-Top6\n(2D-only prediction)", f"3D-Top6\n(3D refinement)"]

plt.boxplot(data, labels=labels, showfliers=False)

# 抖动散点
rng = np.random.RandomState(0)
for i, vals in enumerate(data, start=1):
    if len(vals)==0:
        continue
    xj = i + (rng.rand(len(vals)) - 0.5) * 0.12
    plt.scatter(xj, vals, s=30, alpha=0.75)

plt.ylabel("Stage-2 score")
# plt.title("3D refinement improves prioritization beyond 2D-only Top6")
plt.tight_layout()

out_fig = "figures_jacs/compare_2dTop6_vs_3dTop6_by_3Dscore.png"
plt.savefig(out_fig, dpi=600, bbox_inches="tight")
plt.show()
print("✅ Saved:", out_fig)

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

os.makedirs("tables", exist_ok=True)
os.makedirs("figures_jacs", exist_ok=True)

# -----------------------
# Paths (按你当前文件结构；如有不同只改这里)
# -----------------------
TOP10_3D_PATH = "screening_out/top10_final_from_3d_with_smiles.csv"
TOP1000_3D_PATH = "screening_out/top1000_stage2_3d_scored_with_smiles.csv"  # or *_filtered.csv
TRAIN_PATH = "feat_out/train_with_Xfinal.csv"

assert os.path.exists(TOP10_3D_PATH), f"Missing {TOP10_3D_PATH}"
assert os.path.exists(TOP1000_3D_PATH), f"Missing {TOP1000_3D_PATH}"
assert os.path.exists(TRAIN_PATH), f"Missing {TRAIN_PATH}"

# -----------------------
# Helpers
# -----------------------
def detect_smiles_col(df):
    for c in ["smiles", "SMILES", "canonical_smiles"]:
        if c in df.columns:
            return c
    raise KeyError(f"Cannot find smiles column in df. Columns: {df.columns.tolist()[:30]}")

def canon_inchikey(smiles):
    m = Chem.MolFromSmiles(str(smiles))
    if m is None:
        return None, None
    can = Chem.MolToSmiles(m, canonical=True)
    ik = Chem.inchi.MolToInchiKey(m) if hasattr(Chem, "inchi") else None
    # 如果 RDKit 没编译 inchi，ik 可能为 None；仍可用 canonical smiles 做匹配
    return can, ik

# -----------------------
# 1) Load training set and build membership set
# -----------------------
df_train = pd.read_csv(TRAIN_PATH)
train_sm_col = detect_smiles_col(df_train)

train_can = []
train_ik = []
for s in df_train[train_sm_col].astype(str).tolist():
    can, ik = canon_inchikey(s)
    train_can.append(can)
    train_ik.append(ik)

df_train["_can_smiles"] = train_can
df_train["_inchikey"] = train_ik

# 优先用 InChIKey（更稳），若不可用则退回 canonical smiles
train_set_ik = set([x for x in df_train["_inchikey"].dropna().unique().tolist() if isinstance(x, str)])
train_set_can = set([x for x in df_train["_can_smiles"].dropna().unique().tolist() if isinstance(x, str)])

print("Training unique (InChIKey):", len(train_set_ik), "| unique (canonical smiles):", len(train_set_can))

# -----------------------
# 2) Load Top1000 scored (for补齐) + mark 3D failures if present
# -----------------------
df3 = pd.read_csv(TOP1000_3D_PATH)
sm_col3 = detect_smiles_col(df3)

# 数值化
for c in ["score_3d", "pred_mean_log_3d", "pred_std_log_3d", "score"]:
    if c in df3.columns:
        df3[c] = pd.to_numeric(df3[c], errors="coerce")

# 失败点：mean/std 为 NaN 就视为 3D 失败（与你之前诊断一致）
if "pred_mean_log_3d" in df3.columns and "pred_std_log_3d" in df3.columns:
    df3["is_3d_fail"] = df3["pred_mean_log_3d"].isna() | df3["pred_std_log_3d"].isna()
else:
    df3["is_3d_fail"] = False

# -----------------------
# 3) Canonicalize Top1000 and mark if in training
# -----------------------
can_list = []
ik_list = []
for s in df3[sm_col3].astype(str).tolist():
    can, ik = canon_inchikey(s)
    can_list.append(can)
    ik_list.append(ik)

df3["_can_smiles"] = can_list
df3["_inchikey"] = ik_list

def in_training(row):
    ik = row["_inchikey"]
    cs = row["_can_smiles"]
    if isinstance(ik, str) and len(train_set_ik) > 0:
        return ik in train_set_ik
    if isinstance(cs, str):
        return cs in train_set_can
    return False

df3["in_training_set"] = df3.apply(in_training, axis=1)

# -----------------------
# 4) Pick final 10 synthesis candidates:
#    - highest score_3d
#    - not 3D fail
#    - not in training set (novel)
# -----------------------
df3_rank = df3.sort_values("score_3d", ascending=False, na_position="last").copy()
df3_rank = df3_rank[~df3_rank["is_3d_fail"]].copy()
df3_rank = df3_rank[~df3_rank["in_training_set"]].copy()

final10 = df3_rank.head(10).copy()

# 如果你想“严格去重”（比如不同 SMILES 表示同一个分子），按 InChIKey 去重：
if final10["_inchikey"].notna().any():
    final10 = final10.drop_duplicates(subset=["_inchikey"], keep="first").head(10).copy()
else:
    final10 = final10.drop_duplicates(subset=["_can_smiles"], keep="first").head(10).copy()

# -----------------------
# 5) Save final table
# -----------------------
cols_keep = []
for c in ["lib_idx", sm_col3, "score", "score_3d", "pred_mean_log_3d", "pred_std_log_3d", "is_3d_fail", "in_training_set", "_can_smiles", "_inchikey"]:
    if c in final10.columns:
        cols_keep.append(c)

out_csv = "tables/final_top10_synthesis_candidates.csv"
final10[cols_keep].to_csv(out_csv, index=False, encoding="utf-8")
print("✅ Saved:", out_csv)
display(final10[cols_keep])

# -----------------------
# 6) Draw clear structures (SVG + PNG)
# -----------------------
mols = []
legends = []
for i, r in final10.reset_index(drop=True).iterrows():
    s = r[sm_col3]
    m = Chem.MolFromSmiles(str(s))
    if m is not None:
        m = rdMolDraw2D.PrepareMolForDrawing(m, kekulize=True)
    mols.append(m)
    legends.append(f"#{i+1} idx={int(r['lib_idx']) if 'lib_idx' in r else i}\n3D={float(r['score_3d']):.2f}")

molsPerRow = 5
nRows = (len(mols) + molsPerRow - 1) // molsPerRow
panelW, panelH = 520, 420
W, H = panelW * molsPerRow, panelH * nRows

def _apply_opts(drawer):
    opts = drawer.drawOptions()
    if hasattr(opts, "bondLineWidth"): opts.bondLineWidth = 2.8
    if hasattr(opts, "fixedBondLength"): opts.fixedBondLength = 28
    if hasattr(opts, "legendFontSize"): opts.legendFontSize = 18
    if hasattr(opts, "minFontSize"): opts.minFontSize = 16
    if hasattr(opts, "useBWAtomPalette"): opts.useBWAtomPalette = True
    if hasattr(opts, "padding"): opts.padding = 0.05
    return opts

# SVG
drawer_svg = rdMolDraw2D.MolDraw2DSVG(W, H, panelW, panelH)
_apply_opts(drawer_svg)
drawer_svg.DrawMolecules(mols, legends=legends)
drawer_svg.FinishDrawing()
svg_path = "figures_jacs/final_top10_structures_CLEAR.svg"
with open(svg_path, "w", encoding="utf-8") as f:
    f.write(drawer_svg.GetDrawingText())
print("✅ Saved:", svg_path)

# PNG
drawer_png = rdMolDraw2D.MolDraw2DCairo(W, H, panelW, panelH)
_apply_opts(drawer_png)
drawer_png.DrawMolecules(mols, legends=legends)
drawer_png.FinishDrawing()
png_path = "figures_jacs/final_top10_structures_CLEAR.png"
with open(png_path, "wb") as f:
    f.write(drawer_png.GetDrawingText())
print("✅ Saved:", png_path)

In [ ]:
%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

DATA_PATH = 'Pre.csv'   # Put this notebook in the same folder as Pre(1).csv
OUTDIR = 'figures_pre_ln'
os.makedirs(OUTDIR, exist_ok=True)
